# 01 · Limpieza: raw → interim → processed

Ejecuta el mismo pipeline que `make data` y muestra las validaciones. Si una validación falla, se corrige en `src/optimum/cleaning.py` (con test), nunca a mano en los datos.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from optimum import paths
from optimum.cleaning import load_processed
from optimum.reporting.figures import COLORS, new_figure, save_fig
from optimum.reporting.tables import save_table

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

In [ ]:
from optimum.cleaning import build_processed, save_processed
from optimum.io.excel import export_interim

export_interim()
tables = build_processed()
save_processed(tables)
pd.Series({k: v.shape for k, v in tables.items()}, name="shape")

## Controles

- Cambios de factores: Δ absoluta en tasas; log-retorno en FX y equity (verificado contra `Factor_Changes`).
- Balance inicial: activos 1,000 / pasivos 800 / patrimonio 200.

In [ ]:
inst = tables["instruments"]
inst.groupby("side")["market_value_pen"].sum()

In [ ]:
tables["cash_flows"].groupby("instrument_id").agg(
    n=("period", "size"), ultimo=("payment_date", "max"), principal=("principal_cash_flow", "sum")
)

In [ ]:
# SigmaF del Excel: el caso advierte que puede ser singular
w = np.linalg.eigvalsh(tables["covariance_excel"].values)
pd.Series(w[::-1], name="autovalor").head(8)